# 📜 Constitutional AI: A Per-Rule Critique-and-Revise Loop

## Learning Objectives
In this notebook, you will learn:
1. **A constitution as data** - A short list of named, human-readable rules
2. **A per-rule critique** - One structured `{rule, passes, critique}` verdict per rule, in a single call
3. **Targeted revision** - Rewrite the draft to address only the failing rules, and critique again
4. **A bounded loop with an audit trail** - Stop when every rule passes or at `MAX_ITERATIONS`, and print every verdict along the way

## Where this fits
Module 09 (`modules/09_evaluator_optimizer/notebooks/01_evaluator_optimizer_langgraph.ipynb`) looped a generator against one boolean per quality criterion. Constitutional AI is the same shape with the criteria replaced by **named principles** about how to behave - and with a plain Python loop instead of a graph. `07_reflective_code_generation_agent.ipynb` closes the module with a critique that comes from running code rather than from rules.

## Prerequisites
- `03_reflection_architecture.ipynb` (structured critique and refinement)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📖 Part 1: Constitutional AI as a Critique Loop

Constitutional AI (Anthropic, 2022) trains/aligns a model against a small set of written principles instead of relying only on implicit human preference labels. The core *mechanism* — draft, critique against each principle, revise, repeat — is exactly the **Reflection** pattern from `01_reflection_agents.ipynb`, just with the critique broken into named, auditable rules instead of one freeform review.

This notebook implements that mechanism end to end:

1. Define a small, explicit **constitution** — a list of named rules.
2. **Draft** a response to a prompt chosen to plausibly break at least one rule.
3. **Critique** the draft against *each rule independently*, with a structured `{rule, passes, critique}` verdict per rule.
4. **Revise** the draft to address every failing rule, and repeat the critique — bounded to a small number of iterations.
5. Inspect the before/after response and the full table of per-rule verdicts across iterations.

> **Scope note — read this first.** This is a **teaching-style** critique loop meant to make the Constitutional AI mechanism concrete and inspectable. It is **not** a production safety system: there's no adversarial red-teaming, no calibrated judge model, no logging/monitoring, and no policy sign-off process. Production-grade guardrails, content moderation, and safety policy live in [repository 13 (production and observability)](https://github.com/Sourav692/ai-engineering-demystified-13-production-observability) — this notebook should not be mistaken for that.

| Property | Value |
|---|---|
| Origin | Bai et al. (Anthropic), *Constitutional AI: Harmlessness from AI Feedback* (2022). [arXiv:2212.08073](https://arxiv.org/abs/2212.08073) |

### What we are going to do

- Write a **constitution**: ~5 named rules as plain Python data (name, rule text) — no framework magic, just a list we can iterate over.
- Ask the LLM a question that *invites* overconfident, speculative medical advice — a good stress test for a "no professional advice" / "acknowledge uncertainty" constitution.
- Generate a **draft** response with no constitution applied at all, so it plausibly fails at least one rule.
- Run a **structured critique**: one LLM call that returns a list of `{rule, passes, critique}` verdicts, one entry per rule, using a Pydantic schema so the output is a typed list rather than freeform text.
- If any rule fails, **revise**: feed the draft plus every failing critique back to the LLM and ask for a rewrite that addresses each one, then critique the revision again.
- Bound the loop to **2 iterations** so it terminates even if a rule keeps failing, and print the full verdict history so we can see exactly what changed and why.

---
## 📦 Step 1: Environment Setup

Load the API key and create the chat model through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Drafter, per-rule critic and reviser
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📜 Part 2: The Constitution and the Draft

### 2.1 Define the Constitution

A constitution here is nothing more than a small, human-readable list of named rules. Keeping it as plain data (rather than baking it into one giant prompt string) is what lets us critique **each rule independently** in Part 3, instead of getting one blended "looks fine" judgment.

One rule is different. The original "Be Concise" rule ("tight and focused - no padding") never converged: the critic found *something* to trim in every revision, so the loop always ran into its cap. A concrete word limit, **checked in code**, is both stricter and stable - the same lesson as module 09's evaluator.

In [ ]:
# ============================================================================
# CONSTITUTION: Named, human-readable rules as plain data
# ============================================================================
MAX_WORDS = 150

CONSTITUTION = [
    {
        "name": "Helpful and Direct",
        "rule": "Be helpful and directly answer the question the user actually asked.",
    },
    {
        "name": "No Professional Advice",
        "rule": (
            "Do not give specific medical, legal, or financial advice (e.g. a diagnosis, "
            "a treatment plan, a legal conclusion, or a specific financial recommendation). "
            "Instead, suggest consulting a qualified professional for anything specific to "
            "the user's situation."
        ),
    },
    {
        "name": "No Speculation as Fact",
        "rule": (
            "Avoid presenting speculation, guesses, or unverified possibilities as if they "
            "were established fact."
        ),
    },
    {
        "name": "Be Concise",
        "rule": f"Keep the response under {MAX_WORDS} words.",
        "checked_in_code": True,  # a word count is code's job, not the critic's
    },
    {
        "name": "Acknowledge Uncertainty",
        "rule": (
            "Explicitly acknowledge where the answer is uncertain or depends on details "
            "the user hasn't provided."
        ),
    },
]

for item in CONSTITUTION:
    print(f"- {item['name']}: {item['rule']}")

### 2.2 Draft Response

We pick a prompt that *invites* the model to violate this constitution on the first try: a health question phrased to pull for a confident diagnosis and treatment plan. We deliberately generate the draft with **no constitution in the prompt at all**, so the critique loop has real work to do.

In [ ]:
# ============================================================================
# DRAFT: An unconstrained first answer
# ============================================================================
USER_PROMPT = (
    "My knee has been hurting for two weeks after I started running again. "
    "What's wrong with it and exactly how should I treat it?"
)

draft_prompt = (
    "Answer the user's question as helpfully as you can.\n\n"
    f"User question: {USER_PROMPT}"
)

draft_response = llm.invoke(draft_prompt).content

print("=== DRAFT RESPONSE ===")
print(draft_response)

---
## ⚖️ Part 3: Critique Against Every Rule

Instead of one freeform "is this good?" critique, we ask for a **structured verdict per rule**. `RuleVerdict` captures exactly one rule's outcome (`rule`, `passes`, `critique`); `CritiqueReport` wraps a `verdicts` list, one entry per rule, returned in a single structured-output call. Rules marked `checked_in_code` are left out of the prompt and get their verdict from a word count.

In [ ]:
# ============================================================================
# CRITIQUE SCHEMA: One verdict per rule
# ============================================================================
from typing import List

from pydantic import BaseModel, Field


class RuleVerdict(BaseModel):
    """Pass/fail verdict for a single constitutional rule."""

    rule: str = Field(description="The name of the constitutional rule being judged.")
    passes: bool = Field(description="True if the response satisfies this rule, else False.")
    critique: str = Field(
        description=(
            "A short, specific justification for the verdict. If it fails, explain exactly "
            "what in the response violates the rule."
        )
    )


class CritiqueReport(BaseModel):
    """Full critique: one verdict per rule in the constitution."""

    verdicts: List[RuleVerdict] = Field(
        description="One verdict per constitutional rule, in the same order the rules were given."
    )


critique_model = llm.with_structured_output(CritiqueReport)

print(f"✅ RuleVerdict fields: {list(RuleVerdict.model_fields)}")

In [ ]:
# ============================================================================
# CRITIQUE: Judge the draft against every rule
# ============================================================================
def critique_response(response_text: str, constitution: list[dict]) -> CritiqueReport:
    """Judge `response_text` against every rule in `constitution`, one verdict per rule."""

    llm_rules = [item for item in constitution if not item.get("checked_in_code")]
    rules_block = "\n".join(
        f"{i + 1}. [{item['name']}] {item['rule']}" for i, item in enumerate(llm_rules)
    )

    critique_prompt = (
        "You are a strict constitutional reviewer. Judge the RESPONSE below against EACH "
        "rule listed. For every rule, decide whether the response passes or fails, and give "
        "a short, specific critique justifying that verdict. Return exactly one verdict per "
        "rule, in the same order as the rules, using the `rule` field to name which rule "
        "each verdict is for.\n\n"
        f"RULES:\n{rules_block}\n\n"
        f"RESPONSE:\n{response_text}"
    )

    report = critique_model.invoke(critique_prompt)

    # Rules marked checked_in_code get their verdict from code, not from the critic
    n_words = len(response_text.split())
    for item in constitution:
        if item.get("checked_in_code"):
            report.verdicts.append(RuleVerdict(
                rule=item["name"],
                passes=n_words < MAX_WORDS,
                critique=f"{n_words} words (limit {MAX_WORDS}).",
            ))
    return report


draft_critique = critique_response(draft_response, CONSTITUTION)

print("=== CRITIQUE OF DRAFT (iteration 0) ===")
for verdict in draft_critique.verdicts:
    status = "PASS" if verdict.passes else "FAIL"
    print(f"[{status}] {verdict.rule}: {verdict.critique}")

---
## ✏️ Part 4: Revise Until All Rules Pass (Bounded Loop)

If any rule fails, we feed the current response plus **every failing critique** back to the model and ask for a single rewrite that addresses all of them, then critique the revision again. `MAX_ITERATIONS = 2` guarantees the loop terminates, even if a rule turns out to be unsatisfiable for this prompt.

In [ ]:
# ============================================================================
# REVISE: Rewrite to address every failing rule
# ============================================================================
def revise_response(response_text: str, failing_verdicts: list[RuleVerdict]) -> str:
    """Rewrite `response_text` to address every failing verdict."""

    feedback_block = "\n".join(
        f"- [{v.rule}] {v.critique}" for v in failing_verdicts
    )

    revise_prompt = (
        "Rewrite the RESPONSE below so that it addresses EVERY issue listed in FEEDBACK, "
        "while still answering the original user question. Do not introduce new problems "
        "while fixing these ones.\n\n"
        f"ORIGINAL USER QUESTION:\n{USER_PROMPT}\n\n"
        f"RESPONSE:\n{response_text}\n\n"
        f"FEEDBACK (must all be addressed):\n{feedback_block}"
    )

    return llm.invoke(revise_prompt).content

print("✅ revise_response defined")

In [ ]:
# ============================================================================
# BOUNDED LOOP: critique -> revise until all rules pass or the cap is hit
# ============================================================================
MAX_ITERATIONS = 2

current_response = draft_response
current_critique = draft_critique
history = [{"iteration": 0, "response": current_response, "critique": current_critique}]

for iteration in range(1, MAX_ITERATIONS + 1):
    failing = [v for v in current_critique.verdicts if not v.passes]
    if not failing:
        print(f"All rules pass after iteration {iteration - 1}. Stopping.")
        break

    print(f"--- Iteration {iteration}: revising to address {len(failing)} failing rule(s) ---")
    current_response = revise_response(current_response, failing)
    current_critique = critique_response(current_response, CONSTITUTION)
    history.append({"iteration": iteration, "response": current_response, "critique": current_critique})
else:
    remaining = [v.rule for v in current_critique.verdicts if not v.passes]
    if remaining:
        print(f"Reached MAX_ITERATIONS={MAX_ITERATIONS}. Still failing: {remaining}")

final_response = current_response

print(f"\nIterations recorded: {[h['iteration'] for h in history]}")

---
## 🧾 Part 5: Before vs. After, and the Full Verdict Trail

The value of this pattern isn't the final answer alone - it's that we can point at *exactly which rule* changed the response and *why*. Print the draft, the final response, and every per-rule verdict across iterations.

In [ ]:
# ============================================================================
# BEFORE / AFTER + VERDICT TRAIL
# ============================================================================
print("=" * 60)
print("BEFORE (iteration 0 draft)")
print("=" * 60)
print(history[0]["response"])

print("\n" + "=" * 60)
print(f"AFTER (iteration {history[-1]['iteration']} final)")
print("=" * 60)
print(history[-1]["response"])

print("\n" + "=" * 60)
print("FULL VERDICT TRAIL")
print("=" * 60)
for entry in history:
    print(f"\n--- Iteration {entry['iteration']} ---")
    for v in entry["critique"].verdicts:
        status = "PASS" if v.passes else "FAIL"
        print(f"  [{status}] {v.rule}: {v.critique}")

---
## 📝 Summary

In this notebook, we learned:

### 1. A constitution as data
- Constitutional AI is reflection with a named, itemized rubric: independent, auditable rules instead of one blended critique.

### 2. A per-rule critique
- A typed `List[RuleVerdict]` turns critique into data, so the loop can pick out exactly the failing rules.

### 3. Targeted revision
- Only the failing rules' critiques are fed back, and the revision is critiqued against the whole constitution again.

### 4. A bounded loop with an audit trail
- `MAX_ITERATIONS` guarantees termination and reports what still fails; the verdict history shows which rule changed what.
- This is a teaching pattern, not a safety system: production guardrails need adversarial testing, a calibrated independent judge, human review and monitoring - see [repository 13 (production and observability)](https://github.com/Sourav692/ai-engineering-demystified-13-production-observability).

### Next Steps
- **`07_reflective_code_generation_agent.ipynb`**: a critique that comes from running the code - generate, execute, reflect on the error, repair